## Document intelligence

In [21]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeResult, DocumentContentFormat, AnalyzeDocumentRequest
from dotenv import load_dotenv
import os
import base64
import json
load_dotenv()

endpoint = os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"]
key = os.environ["DOCUMENTINTELLIGENCE_API_KEY"]

document_intelligence_client = DocumentIntelligenceClient(
    endpoint=endpoint, credential=AzureKeyCredential(key),
    api_version="2024-11-30"  
)

filepath = "../datasets/how-to-pdfs/how-to-guides.docx"
with open(filepath, "rb") as f:
    poller = document_intelligence_client.begin_analyze_document(
        "prebuilt-layout",
        AnalyzeDocumentRequest(bytes_source=f.read()),
        output_content_format=DocumentContentFormat.MARKDOWN,
    )
    result = poller.result()

with open("layout-results-markdown.json", "w") as f:
    json.dump(result.as_dict(), f, indent=2)

# with open("layout-results-markdown.md", "w") as f:
#     f.write(result.content)


In [24]:
# Load the layout analysis result from JSON
with open("layout-results-markdown.json", "r") as f:
    layout_data = json.load(f)

    paragraphs = layout_data.get("paragraphs", [])
    for idx, para in enumerate(paragraphs):
        if "role" in para:
            print(f"Index: {idx}, Role: {para['role']}, Value: {para.get('content', '')}")

Index: 0, Role: sectionHeading, Value: Customer Service Automation
Index: 31, Role: sectionHeading, Value: Product Discovery
Index: 62, Role: sectionHeading, Value: Account and Security
Index: 93, Role: sectionHeading, Value: Seller Support


## Pandas Exersize

In [ ]:

import pandas as pd

# Read the CSV file
df = pd.read_csv('../datasets/csv/customer_support_tickets.csv')

# Transform column names: lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')


# Display the transformed dataframe
df.head()

,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [3]:
# Replace the placeholder {product_purchased} with actual values from the product_purchased column
df['ticket_description'] = df.apply(
    lambda row: row['ticket_description'].replace('{product_purchased}', str(row['product_purchased'])), 
    axis=1
)


In [4]:
df.head()

,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the GoPro Hero. Pleas...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the LG Smart TV. Plea...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my Dell XPS. The Del...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the Microsoft Office....,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the Autodesk AutoCAD....,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [5]:
# Split dataframe into two parts
df_first_10 = df.head(10)  # First 10 rows
df_rest = df.iloc[10:]      # All rows from index 10 onwards

print(f"First dataframe: {len(df_first_10)} rows")
print(f"Second dataframe: {len(df_rest)} rows")

First dataframe: 10 rows
Second dataframe: 8459 rows


In [6]:
# Save the transformed data (optional)
df_first_10.to_csv('../datasets/csv/customer_support_tickets_part1.csv', index=False)
df_rest.to_csv('../datasets/csv/customer_support_tickets_part2.csv', index=False)

In [7]:
print(df_first_10.columns)

Index(['ticket_id', 'customer_name', 'customer_email', 'customer_age',
       'customer_gender', 'product_purchased', 'date_of_purchase',
       'ticket_type', 'ticket_subject', 'ticket_description', 'ticket_status',
       'resolution', 'ticket_priority', 'ticket_channel',
       'first_response_time', 'time_to_resolution',
       'customer_satisfaction_rating'],
      dtype='object')
